In [ ]:
import pascal_dataset
import torchvision.transforms as T
from torchvision import transforms

In [ ]:
encode_transform = transforms.Compose(
    [
        transforms.Resize((256,256), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.ToTensor(),
    ]
)
transform = T.Compose([
        T.Resize((224, 224)),
        encode_transform  # Normalization and tensor
    ])


In [ ]:
data = pascal_dataset.DatasetPASCAL(datapath='/path/to/data', fold=0,image_transform=transform, mask_transform=transform,support=4)

In [ ]:
data[0]

In [ ]:
size = data.__len__()
print(f'the lenght is {size}')
data[0]

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T
import numpy as np

# Get one sample
sample = data[0]

support_imgs = sample["support_images"]
support_masks = sample["support_masks"]
query_img = sample["query_image"]
query_mask = sample["query_mask"]

to_pil = T.ToPILImage()

num_support = len(support_imgs)

# ---------------------------
# Plot support pairs
# ---------------------------
fig, axes = plt.subplots(num_support, 2, figsize=(6, 3 * num_support))

if num_support == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(num_support):
    img = support_imgs[i]
    mask = support_masks[i]

    # Convert to PIL
    img_pil = to_pil(img)
    mask_pil = to_pil(mask).convert("RGB")

    axes[i, 0].imshow(img_pil)
    axes[i, 0].set_title(f"Support Image {i}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_pil)
    axes[i, 1].set_title(f"Support Mask {i}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

# ---------------------------
# Plot query image & mask
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].imshow(to_pil(query_img))
axes[0].set_title("Query Image")
axes[0].axis("off")

axes[1].imshow(to_pil(query_mask))
axes[1].set_title("Query Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()



In [ ]:
plt.imshow(sample['target'][0], cmap='gray')

In [ ]:
image_np = sample['target'].squeeze(0).cpu().numpy()
binary_mask = (image_np >= 0.5).astype(np.uint8)
plt.imshow(binary_mask, cmap='gray')

In [ ]:
image_pil=T.ToPILImage()(sample['target'].cpu())
image_pil=np.asarray(image_pil)
image_pil = (image_pil >= 125).astype(np.uint8)


In [ ]:
lvm_path='/path/to/lvm_ckpt'  # path to converted hf model
vqgan_path='/path/to/vqgan'  # path to vqgan model

In [ ]:
from transformers import AutoModel, GenerationConfig
from model_hf.muse import VQGANModel
from utils import convert_decode_to_pil, encode_transform

In [ ]:
model = AutoModel.from_pretrained(lvm_path, trust_remote_code=True).cuda().eval()
vq_model = VQGANModel.from_pretrained(vqgan_path).cuda().eval()

generation_config = GenerationConfig(
        temperature=0.1,
        top_p=0.75,
        num_beams=1,
        early_stopping=True,
        max_new_tokens=256,
    )

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
input_images = sample['input']
query_img = input_images[-1]
query_name = sample['query_name']

# # Prompt images (support set)
# support_imgs = [input_images[i] for i in [0, 4]]  # 2-shot
# Prompt images (support set)
n_supports = (len(input_images) - 1) // 2  # each support: (img, mask)
support_imgs = [input_images[i * 2] for i in range(n_supports)]  # support images only

prompt_ids = []
for s_img in support_imgs:
    s_img = s_img[0:3,:,:].unsqueeze(0).to(device)
    _, indices = vq_model.encode(s_img)
    prompt_ids.append(indices.view(1, -1))

seq_prompt_ids = torch.cat(prompt_ids, dim=1)

# Encode query image
query_img = query_img.unsqueeze(0).to(device)
_, q_indices = vq_model.encode(query_img)
input_ids = torch.cat([seq_prompt_ids, q_indices.view(1, -1)], dim=1)

In [ ]:
# Inference
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        generation_config=generation_config,
        max_new_tokens=256,
        return_dict_in_generate=True
    )

# Decode
generated_tokens = vq_model.quantize.get_codebook_entry_for_lvm(outputs.sequences[:, -256:])
generated_img = vq_model.decode(
generated_tokens.view(1, generated_tokens.shape[1] // 16, 16, -1).permute(0, 3, 1, 2)
)
generated_img_pil = convert_decode_to_pil(generated_img)[0]

In [ ]:
plt.imshow(generated_img_pil)
plt.axis('off')
plt.title('Generated Image')
plt.show()

In [ ]:
import numpy as np
from PIL import Image
# Display generated image
# Convert to grayscale
gray = generated_img_pil.convert('L')  # 'L' mode = grayscale

# Convert to NumPy array
gray_np = np.array(gray)

# Apply binary threshold
threshold = 90  # you can adjust this value
binary_np = (gray_np > threshold).astype(np.uint8)   # 0 or 255

# Convert back to PIL image
binary_img = Image.fromarray(binary_np)

# Show or save result
plt.imshow(binary_img, cmap='gray')

In [ ]:
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)

def round_image(img, options=(WHITE, BLACK), outputs=None, t=(0, 0, 0)):
    # img.shape == [224, 224, 3], img.dtype == torch.int32
    img = torch.tensor(img)
    t = torch.tensor((t)).to(img)
    options = torch.tensor(options)
    opts = options.view(len(options), 1, 1, 3).permute(1, 2, 3, 0).to(img)
    nn = (((img + t).unsqueeze(-1) - opts) ** 2).float().mean(dim=2)
    nn_indices = torch.argmin(nn, dim=-1)
    if outputs is None:
        outputs = options
    res_img = torch.tensor(outputs)[nn_indices]
    return res_img

In [ ]:
generated_img_pil = np.array(generated_img_pil)

In [ ]:
round_img= round_image(generated_img_pil, options=(WHITE, BLACK), outputs=None, t=(0, 0, 0))

In [ ]:
plt.imshow(round_img)
plt.axis('off')